# Notebook 1 — Modelo ARIMA / SARIMA

**Proyecto:** Sistema de pronóstico de demanda eléctrica en Ecuador

Este notebook implementa el primer modelo del estudio comparativo: la familia ARIMA/SARIMA. La metodología sigue el procedimiento de Box-Jenkins con búsqueda en grilla optimizada por AIC.

**Pasos:**
1. Cargar `train.csv`, `val.csv`, `test.csv` generados en Notebook 0
2. Identificar parámetros (p, d, q) y (P, D, Q, s) por búsqueda en grilla
3. Ajustar el mejor modelo sobre train+val
4. Predicción sobre el conjunto de prueba
5. Cálculo de métricas: MAE, RMSE, MAPE
6. Visualización y exportación de resultados

**Requisito:** ejecutar primero Notebook 0.


## 1. Importaciones

In [ ]:
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["font.family"] = "serif"

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def reportar(y_true, y_pred, etiqueta=""):
    mae_v  = mean_absolute_error(y_true, y_pred)
    rmse_v = rmse(y_true, y_pred)
    mape_v = mape(y_true, y_pred)
    print(f"{etiqueta:20s}  MAE={mae_v:8.2f}  RMSE={rmse_v:8.2f}  MAPE={mape_v:6.2f}%")
    return {"MAE": mae_v, "RMSE": rmse_v, "MAPE": mape_v}


## 2. Carga de los datos preprocesados

In [ ]:
train = pd.read_csv("train.csv", parse_dates=["fecha"]).set_index("fecha")["demanda_gwh"].asfreq("MS")
val   = pd.read_csv("val.csv",   parse_dates=["fecha"]).set_index("fecha")["demanda_gwh"].asfreq("MS")
test  = pd.read_csv("test.csv",  parse_dates=["fecha"]).set_index("fecha")["demanda_gwh"].asfreq("MS")

print(f"Train: {len(train)} obs  [{train.index.min().date()} -> {train.index.max().date()}]")
print(f"Val:   {len(val)} obs    [{val.index.min().date()} -> {val.index.max().date()}]")
print(f"Test:  {len(test)} obs   [{test.index.min().date()} -> {test.index.max().date()}]")


## 3. Búsqueda en grilla SARIMA optimizada por AIC

Se exploran combinaciones de:
- p, d, q ∈ {0, 1, 2}
- P, D, Q ∈ {0, 1} con s=12

Esto da hasta 108 modelos. Se selecciona el de menor AIC sobre el conjunto train+val sin el test.


In [ ]:
train_val = pd.concat([train, val])

p_vals = d_vals = q_vals = [0, 1, 2]
P_vals = D_vals = Q_vals = [0, 1]
s = 12

resultados = []
combos = list(itertools.product(p_vals, d_vals, q_vals, P_vals, D_vals, Q_vals))
print(f"Total de combinaciones a evaluar: {len(combos)}\n")

for i, (p, d, q, P, D, Q) in enumerate(combos):
    try:
        mod = SARIMAX(train_val, order=(p, d, q), seasonal_order=(P, D, Q, s),
                      enforce_stationarity=False, enforce_invertibility=False)
        res = mod.fit(disp=False, maxiter=200)
        resultados.append({"p": p, "d": d, "q": q, "P": P, "D": D, "Q": Q,
                          "AIC": res.aic, "BIC": res.bic})
    except Exception:
        continue

df_grid = pd.DataFrame(resultados).sort_values("AIC").reset_index(drop=True)
print(f"Modelos ajustados con éxito: {len(df_grid)}\n")
print("TOP 10 modelos por AIC:")
print(df_grid.head(10).to_string(index=False))


## 4. Ajuste del mejor modelo SARIMA

In [ ]:
mejor = df_grid.iloc[0]
order = (int(mejor.p), int(mejor.d), int(mejor.q))
seasonal_order = (int(mejor.P), int(mejor.D), int(mejor.Q), s)
print(f"Mejor modelo: SARIMA{order} x {seasonal_order}")
print(f"AIC = {mejor.AIC:.2f},  BIC = {mejor.BIC:.2f}")

# Ajuste final sobre train+val
modelo_final = SARIMAX(train_val, order=order, seasonal_order=seasonal_order,
                       enforce_stationarity=False, enforce_invertibility=False)
resultado_final = modelo_final.fit(disp=False, maxiter=300)
print("\nResumen del modelo:")
print(resultado_final.summary().tables[1])


## 5. Diagnóstico de residuos

In [ ]:
fig = resultado_final.plot_diagnostics(figsize=(13, 8))
plt.tight_layout()
plt.savefig("sarima_diagnosticos.png", dpi=300, bbox_inches="tight")
plt.show()

# Prueba de Ljung-Box (residuos blancos = buen ajuste)
lb = acorr_ljungbox(resultado_final.resid, lags=[12], return_df=True)
print("Prueba de Ljung-Box sobre los residuos (H0: residuos no correlacionados):")
print(lb)
print("\n=> Si p-valor > 0.05, los residuos se comportan como ruido blanco (buen ajuste).")


## 6. Predicción sobre el conjunto de prueba

In [ ]:
pred = resultado_final.get_forecast(steps=len(test))
y_pred_test = pred.predicted_mean
intervalo = pred.conf_int(alpha=0.05)
y_pred_test.index = test.index
intervalo.index = test.index

# También guardamos predicción in-sample para gráfico completo
pred_in_sample = resultado_final.get_prediction(start=train_val.index[12], end=train_val.index[-1])
y_pred_train = pred_in_sample.predicted_mean

print("Predicciones sobre TEST:")
comp = pd.DataFrame({
    "Real (GWh)": test.values,
    "Predicho (GWh)": y_pred_test.values,
    "Error": test.values - y_pred_test.values,
    "Error %": ((test.values - y_pred_test.values) / test.values) * 100
}, index=test.index)
print(comp.round(2).to_string())


## 7. Cálculo de métricas

In [ ]:
print("Desempeño del modelo SARIMA:\n")
m_train = reportar(train_val.iloc[12:], y_pred_train, "Train+Val (in-sample)")
m_test  = reportar(test, y_pred_test,                  "Test (out-of-sample)")


## 8. Gráfico de predicción

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
train.plot(ax=ax, label="Train", color="#1f4e79", lw=1.5)
val.plot(ax=ax,   label="Val",   color="#2e7d32", lw=1.5)
test.plot(ax=ax,  label="Test (real)", color="#c62828", lw=1.8, marker="o", ms=5)
y_pred_test.plot(ax=ax, label="SARIMA (pred.)", color="#ef6c00", lw=2.0, marker="s", ms=5, linestyle="--")
ax.fill_between(test.index, intervalo.iloc[:, 0], intervalo.iloc[:, 1],
                color="#ef6c00", alpha=0.18, label="IC 95%")
ax.set_title(f"SARIMA{order} x {seasonal_order}: predicción del conjunto de prueba")
ax.set_ylabel("Demanda (GWh)"); ax.set_xlabel("Fecha"); ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_sarima_prediccion.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Exportación de resultados

In [ ]:
resultados_sarima = pd.DataFrame({
    "fecha": test.index,
    "real_gwh": test.values,
    "sarima_pred_gwh": y_pred_test.values,
    "sarima_ic_lower": intervalo.iloc[:, 0].values,
    "sarima_ic_upper": intervalo.iloc[:, 1].values,
})
resultados_sarima.to_csv("resultados_sarima.csv", index=False)

metricas_sarima = pd.DataFrame([
    {"modelo": "SARIMA", "conjunto": "train+val", **m_train},
    {"modelo": "SARIMA", "conjunto": "test",      **m_test},
])
metricas_sarima.to_csv("metricas_sarima.csv", index=False)

# Guardamos también la configuración del modelo
import json
config = {
    "modelo": "SARIMA",
    "order": list(order),
    "seasonal_order": list(seasonal_order),
    "AIC": float(mejor.AIC),
    "BIC": float(mejor.BIC),
    "Ljung_Box_p": float(lb["lb_pvalue"].iloc[0])
}
with open("config_sarima.json", "w") as f:
    json.dump(config, f, indent=2)

print("Exportado:")
print("  resultados_sarima.csv   (predicciones e intervalos sobre test)")
print("  metricas_sarima.csv     (MAE, RMSE, MAPE)")
print("  config_sarima.json      (parámetros y diagnóstico)")
print("  fig_sarima_prediccion.png")
print("  sarima_diagnosticos.png")
print("\nNotebook 1 completado. Continúa con Notebook 2 (Prophet).")
